# Super-Wised Coding LLM 1.0 — Kaggle smoke test

This notebook validates the pipeline with a small model. It does not train a 100M or 10B production model. Attach the project ZIP as a Kaggle input or upload the folder. Review data licenses before collecting any corpus.

In [ ]:
import glob, os, zipfile, pathlib
candidates = glob.glob('/kaggle/input/**/*.zip', recursive=True)
project = pathlib.Path('/kaggle/working/super-wised-code-llm-1.0')
if not project.exists():
    if not candidates:
        raise FileNotFoundError('Attach super-wised-code-llm-1.0.zip as a Kaggle Notebook input, or upload/extract the project folder into /kaggle/working.')
    with zipfile.ZipFile(candidates[0]) as zf:
        zf.extractall('/kaggle/working')
    print('Extracted:', candidates[0])
os.chdir(project)
print('Project:', os.getcwd())
print('GPU available:', __import__('torch').cuda.is_available())

## 1. Install only the project requirements

If Kaggle reports unrelated preinstalled-package resolver warnings, do not blindly upgrade/downgrade every package. Restart the session if you changed core packages and inspect the actual import error.

In [ ]:
%pip install -r requirements.txt

from pathlib import Path
import json
Path('data').mkdir(exist_ok=True)
raw = Path('data/raw.jsonl')
if not raw.exists():
    with raw.open('w', encoding='utf-8') as f:
        for i in range(1200):
            record = {'text': f'export function calculateTotal{i}(price, quantity) {{\n  const total = Number(price) * Number(quantity);\n  if (!Number.isFinite(total)) throw new Error(\"Invalid total\");\n  return total;\n}}\n// Example {i}: validate numeric inputs before use.', 'license': 'MIT', 'source': 'synthetic-demo', 'path': f'src/example{i}.js'}
            f.write(json.dumps(record) + '\n')
print('Demo rows:', sum(1 for _ in raw.open(encoding='utf-8')))
print('Synthetic-only data: replace with licensed, reviewed source code for real training.')

In [ ]:
from pathlib import Path
Path('data').mkdir(exist_ok=True)
if not Path('data/raw.jsonl').exists():
    Path('data/raw.jsonl').write_text('{"text":"export function add(a, b) { return a + b; }\n","license":"MIT","source":"synthetic-demo"}\n', encoding='utf-8')
print('Input exists:', Path('data/raw.jsonl').exists())

In [ ]:
This notebook generates a tiny synthetic corpus only to test the mechanics. It is not suitable for teaching a coding model accurately. Replace it with a properly licensed and quality-reviewed corpus before evaluating model capability.


## 3. Train and verify checkpointing

The synthetic demo is too small to train; replace it with enough licensed code so both splits contain more than `block_size + 1` tokens. Run the smoke configuration first.

In [ ]:
!python -m src.train --config configs/kaggle-smoke.json --data-dir data/processed --output-dir runs/smoke

In [ ]:
!python -m src.train --config configs/kaggle-smoke.json --data-dir data/processed --output-dir runs/smoke --resume

## 4. Evaluate and prepare a release

Publishing is optional. Review the output and model card; a smoke-test checkpoint must not be advertised as a production-quality coding model.

In [ ]:
!python -m src.evaluate --config configs/kaggle-smoke.json --data-dir data/processed --checkpoint runs/smoke/best.pt

In [ ]:
!python -m src.publish --checkpoint runs/smoke/best.pt --config configs/kaggle-smoke.json --repo-id YOUR_HF_USERNAME/super-wised-code-llm-1.0 --output-dir release